# ECE 175B ADG — Kaggle T4 Training

> Spring 2026 final project. Train Attribute-Disentangled Guidance on CelebA 64×64.
>
> **Time budget**: ~6h on T4 GPU (25 epochs). Kaggle session limit 12h — should fit.
>
> **Dataset**: Must add Kaggle CelebA dataset input before running.
> Go to **Add Data** → search "celeba-dataset" → use jessicali9530/celeba-dataset.

## Cell 1: Install dependencies

In [ ]:
!pip install -q diffusers>=0.27 accelerate>=0.27 tqdm
print("Dependencies installed.")

## Cell 2: Upload project code

Upload these files to `/kaggle/working/`:
- `data.py`
- `model.py`
- `ddpm.py`
- `adg.py`
- `cfg.py`
- `train.py`
- `sample.py`

Or clone from GitHub if you've pushed the repo.

In [ ]:
# Option 1: Manual upload via Kaggle UI (drag-drop 7 .py files to /kaggle/working/)
# Option 2: Clone from GitHub
# !git clone https://github.com/<your-username>/ece175b-adg.git /kaggle/working/adg
# %cd /kaggle/working/adg

# For now, assume files are uploaded to /kaggle/working/
import os
os.chdir('/kaggle/working')
print("Working directory:", os.getcwd())
print("Files:", os.listdir('.'))

## Cell 3: Adapt data.py for Kaggle CelebA path

Kaggle CelebA dataset path: `/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba/`

We need to adapt `data.py` to use this path. Create a small wrapper class.

In [ ]:
# Create kaggle_data.py — wrapper around data.py with Kaggle path
kaggle_data_code = '''
"""Kaggle-specific data wrapper for CelebA.

Kaggle CelebA dataset structure:
/kaggle/input/celeba-dataset/
  img_align_celeba/img_align_celeba/  <- images
  list_attr_celeba.csv                <- attributes

torchvision CelebA expects:
  <root>/celeba/img_align_celeba/     <- images
  <root>/celeba/list_attr_celeba.txt  <- attributes

So we need to manually point to Kaggle paths.
"""
from pathlib import Path
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import pandas as pd

# Kaggle CelebA paths
KAGGLE_IMG_DIR = Path("/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba")
KAGGLE_ATTR_CSV = Path("/kaggle/input/celeba-dataset/list_attr_celeba.csv")

ATTR_NAMES = [
    "5_o_Clock_Shadow", "Arched_Eyebrows", "Attractive", "Bags_Under_Eyes", "Bald",
    "Bangs", "Big_Lips", "Big_Nose", "Black_Hair", "Blond_Hair", "Blurry",
    "Brown_Hair", "Bushy_Eyebrows", "Chubby", "Double_Chin", "Eyeglasses",
    "Goatee", "Gray_Hair", "Heavy_Makeup", "High_Cheekbones", "Male",
    "Mouth_Slightly_Open", "Mustache", "Narrow_Eyes", "No_Beard", "Oval_Face",
    "Pale_Skin", "Pointy_Nose", "Receding_Hairline", "Rosy_Cheeks", "Sideburns",
    "Smiling", "Straight_Hair", "Wavy_Hair", "Wearing_Earrings", "Wearing_Hat",
    "Wearing_Lipstick", "Wearing_Necklace", "Wearing_Necktie", "Young",
]
ATTR_NAME_TO_IDX = {n.lower(): i for i, n in enumerate(ATTR_NAMES)}

def attr_indices(attrs):
    return [ATTR_NAME_TO_IDX[a.lower()] for a in attrs]

class KaggleCelebASubset(Dataset):
    def __init__(self, attrs=None, resolution=64, split="train"):
        attrs = attrs or ["smiling", "eyeglasses", "male", "young"]
        self.attrs = attrs
        self.attr_idx = attr_indices(attrs)
        self.resolution = resolution
        
        # Load attribute CSV
        df = pd.read_csv(KAGGLE_ATTR_CSV)
        # CSV columns: image_id, attr1, attr2, ... (40 attrs)
        # Values: 1 or -1 → convert to 0/1
        self.image_names = df.iloc[:, 0].tolist()
        attr_cols = df.columns[1:]
        attr_matrix = df[attr_cols].values  # (N, 40)
        attr_matrix = (attr_matrix + 1) // 2  # -1→0, 1→1
        self.attr_matrix = torch.tensor(attr_matrix, dtype=torch.float32)
        
        # Split: use first 80% for train, last 20% for val (simple split)
        n = len(self.image_names)
        if split == "train":
            self.indices = list(range(int(n * 0.8)))
        else:
            self.indices = list(range(int(n * 0.8), n))
        
        self.transform = transforms.Compose([
            transforms.Resize(resolution),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3),
        ])
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        img_name = self.image_names[real_idx]
        img_path = KAGGLE_IMG_DIR / img_name
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        
        attr = self.attr_matrix[real_idx][self.attr_idx]
        return {"image": img, "attr": attr}

def get_kaggle_dataloader(attrs=None, resolution=64, batch_size=128, split="train", num_workers=2, shuffle=True):
    ds = KaggleCelebASubset(attrs=attrs, resolution=resolution, split=split)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers,
        pin_memory=torch.cuda.is_available(), drop_last=True,
    )
'''

with open('kaggle_data.py', 'w') as f:
    f.write(kaggle_data_code)
print("kaggle_data.py created.")

## Cell 4: Verify GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be extremely slow.")

## Cell 5: Smoke test data loading

In [ ]:
from kaggle_data import get_kaggle_dataloader

loader = get_kaggle_dataloader(
    attrs=["smiling", "eyeglasses", "male", "young"],
    resolution=64,
    batch_size=4,
    split="train",
    num_workers=2,
    shuffle=False,
)
print(f"Dataset size: {len(loader.dataset)}")
batch = next(iter(loader))
print(f"Image batch: {batch['image'].shape}, range [{batch['image'].min():.2f}, {batch['image'].max():.2f}]")
print(f"Attr batch: {batch['attr'].shape}, sample: {batch['attr'][0].tolist()}")
print("Data loading OK.")

## Cell 6: Training

**Time budget**: ~6h for 25 epochs on T4 (16GB VRAM, batch_size=128).

Checkpoint saved every 5 epochs to `/kaggle/working/checkpoints/`.

In [ ]:
import os
from pathlib import Path
import torch
from accelerate import Accelerator
from tqdm import tqdm

from kaggle_data import get_kaggle_dataloader
from ddpm import make_scheduler, training_step
from model import AttrConditionedUNet

# Hyperparameters (Kaggle-optimized)
ATTRS = ["smiling", "eyeglasses", "male", "young"]
RESOLUTION = 64
BATCH_SIZE = 128  # T4 16GB can handle this
EPOCHS = 25       # Reduced from 50 to fit 6h budget
LR = 2e-4
ATTR_DROP_P = 0.1
SAVE_DIR = "/kaggle/working/checkpoints"
SAVE_EVERY = 5
NUM_WORKERS = 2   # Kaggle kernel has 2 CPUs
MIXED_PRECISION = "fp16"

Path(SAVE_DIR).mkdir(exist_ok=True, parents=True)

accelerator = Accelerator(mixed_precision=MIXED_PRECISION)
device = accelerator.device
n_attrs = len(ATTRS)

# Data
loader = get_kaggle_dataloader(
    attrs=ATTRS, resolution=RESOLUTION, batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS, shuffle=True, split="train"
)

# Model + scheduler
model = AttrConditionedUNet(sample_size=RESOLUTION, n_attrs=n_attrs)
scheduler = make_scheduler()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

model, optimizer, loader = accelerator.prepare(model, optimizer, loader)

print(f"Device: {device}, mixed_precision: {MIXED_PRECISION}")
print(f"Attributes (K={n_attrs}): {ATTRS}")
print(f"Dataset size: {len(loader.dataset)}, batch size: {BATCH_SIZE}")
n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params/1e6:.2f}M")
print(f"Training {EPOCHS} epochs (wall-clock budget: ~6h on T4)\n")

global_step = 0
for epoch in range(EPOCHS):
    model.train()
    pbar = tqdm(loader, disable=not accelerator.is_main_process, desc=f"epoch {epoch}")
    epoch_losses = []
    for batch in pbar:
        optimizer.zero_grad()
        loss = training_step(model, scheduler, batch, device, attr_drop_p=ATTR_DROP_P)
        accelerator.backward(loss)
        optimizer.step()
        global_step += 1
        epoch_losses.append(loss.item())
        if global_step % 100 == 0:
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    
    if accelerator.is_main_process:
        avg = sum(epoch_losses) / max(len(epoch_losses), 1)
        print(f"  epoch {epoch} done. avg loss: {avg:.4f}")
        
        if (epoch + 1) % SAVE_EVERY == 0 or epoch == EPOCHS - 1:
            ckpt_path = Path(SAVE_DIR) / f"ckpt_epoch{epoch + 1:03d}.pt"
            state = {
                "model": accelerator.unwrap_model(model).state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
                "attrs": ATTRS,
                "resolution": RESOLUTION,
            }
            torch.save(state, ckpt_path)
            # Save best.pt (latest)
            best_path = Path(SAVE_DIR) / "best.pt"
            torch.save(state, best_path)
            print(f"  Saved → {ckpt_path}")

print("\nTraining done.")
print(f"Checkpoint saved to {SAVE_DIR}/best.pt")

## Cell 7: Sampling — CFG baseline

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision
from pathlib import Path

from cfg import cfg_cond_fn
from ddpm import ddpm_sample_loop, make_scheduler
from model import AttrConditionedUNet

CKPT_PATH = "/kaggle/working/checkpoints/best.pt"
RESULT_DIR = "/kaggle/working/results"
Path(RESULT_DIR).mkdir(exist_ok=True, parents=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_attrs = 4
resolution = 64

# Load model
model = AttrConditionedUNet(sample_size=resolution, n_attrs=n_attrs).to(device)
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()
scheduler = make_scheduler()

# Target: [smiling=1, eyeglasses=1, male=0, young=1]
target = torch.tensor([[1.0, 1.0, 0.0, 1.0]], device=device).repeat(16, 1)

# CFG with w=4.0
torch.manual_seed(0)
cond = cfg_cond_fn(target, w=4.0)
imgs = ddpm_sample_loop(model, scheduler, (16, 3, 64, 64), device, cond)

# Visualize
grid = torchvision.utils.make_grid(imgs, nrow=4, normalize=True, value_range=(-1, 1))
grid = grid.permute(1, 2, 0).cpu().numpy()

plt.figure(figsize=(8, 8))
plt.imshow(grid)
plt.axis("off")
plt.title("CFG (w=4.0) — target=[smiling, eyeglasses, ~male, young]", fontsize=10)
plt.tight_layout()
plt.savefig(f"{RESULT_DIR}/cfg_baseline.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved → {RESULT_DIR}/cfg_baseline.png")

## Cell 8: Sampling — ADG with per-attribute w_k

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision

from adg import adg_cond_fn
from ddpm import ddpm_sample_loop, make_scheduler

# ADG: w_smiling=1, w_eyeglasses=4 (strong), w_male=0, w_young=1
ws = [1.0, 4.0, 0.0, 1.0]
torch.manual_seed(0)
cond = adg_cond_fn(target, ws)
imgs = ddpm_sample_loop(model, scheduler, (16, 3, 64, 64), device, cond)

grid = torchvision.utils.make_grid(imgs, nrow=4, normalize=True, value_range=(-1, 1))
grid = grid.permute(1, 2, 0).cpu().numpy()

plt.figure(figsize=(8, 8))
plt.imshow(grid)
plt.axis("off")
plt.title(f"ADG (w=[1, 4, 0, 1]) — strong eyeglasses", fontsize=10)
plt.tight_layout()
plt.savefig(f"{RESULT_DIR}/adg_strong_eyeglasses.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved → {RESULT_DIR}/adg_strong_eyeglasses.png")

## Cell 9: ADG sweep — vary w_eyeglasses from 0 to 6

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision

# Sweep w_eyeglasses: 0 → 6 (7 steps), fix others at w=1
sweep_values = torch.linspace(0, 6, 7)
all_imgs = []

for v in sweep_values:
    ws = [1.0, float(v), 0.0, 1.0]  # vary w_eyeglasses
    torch.manual_seed(0)  # same seed for comparison
    cond = adg_cond_fn(target, ws)
    imgs = ddpm_sample_loop(model, scheduler, (4, 3, 64, 64), device, cond)  # 4 imgs per step
    all_imgs.append(imgs)
    print(f"w_eyeglasses={v:.1f} done")

imgs_concat = torch.cat(all_imgs, dim=0)  # (28, 3, 64, 64)
grid = torchvision.utils.make_grid(imgs_concat, nrow=4, normalize=True, value_range=(-1, 1))
grid = grid.permute(1, 2, 0).cpu().numpy()

plt.figure(figsize=(10, 12))
plt.imshow(grid)
plt.axis("off")
plt.title("ADG sweep: w_eyeglasses 0→6 (rows: w=0, 1, 2, 3, 4, 5, 6)", fontsize=10)
plt.tight_layout()
plt.savefig(f"{RESULT_DIR}/adg_sweep_eyeglasses.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved → {RESULT_DIR}/adg_sweep_eyeglasses.png")

## Cell 10: Download results

Kaggle auto-saves `/kaggle/working/` to output when session ends.

Or manually download:
- `/kaggle/working/checkpoints/best.pt` (~100 MB)
- `/kaggle/working/results/*.png`

In [ ]:
import os
print("Results in /kaggle/working/results/:")
for f in os.listdir("/kaggle/working/results"):
    print(f"  {f}")
print("\nCheckpoints in /kaggle/working/checkpoints/:")
for f in os.listdir("/kaggle/working/checkpoints"):
    size = os.path.getsize(f"/kaggle/working/checkpoints/{f}") / 1e6
    print(f"  {f} ({size:.1f} MB)")